## Setup

*You must run the cells in this section each time you connect to a new runtime. For example, when you return to the notebook after an idle timeout, when the runtime crashes, or when you restart or factory reset the runtime.*

Install requirements:

In [ ]:
! pip install --upgrade pip > pip.log
! pip install --upgrade ocdskingfishercolab psycopg2-binary >> pip.log

In [ ]:
# @title Import packages and load extensions { display-mode: "form" }

import gzip
import json
import os
import shutil
import tempfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta
from google.colab.data_table import DataTable
from google.colab.files import download
from ipywidgets import widgets
from ocdskingfishercolab import (
    authenticate_gspread,
    calculate_coverage,
    check_usability_indicators,
    download_dataframe_as_csv,
    format_coverage,
    format_thousands,
    get_publication_select_box,
    get_publications,
    indicator_checks,
    load_indicators,
    most_common_fields_to_calculate_indicators,
    plot_objects_per_stage,
    plot_objects_per_year,
    plot_release_count,
    plot_releases_by_month,
    plot_top_buyers,
    plot_usability_indicators,
    render_json,
    save_dataframe_to_sheet,
    save_dataframe_to_spreadsheet,
    set_dark_mode,
    set_light_mode,
)

# Load https://pypi.org/project/ipython-sql/
%load_ext sql
# Load https://colab.research.google.com/notebooks/data_table.ipynb
%load_ext google.colab.data_table

In [ ]:
# @title Configure the notebook environment { display-mode: "form" }

# Increase max columns so that Pandas DataFrames with many columns are rendered as data tables.
DataTable.max_columns = 50
# Remove the index from data tables for easier copy-pasting to Google Docs.
DataTable.include_index = False

# Return Pandas DataFrames instead of regular result sets.
%config SqlMagic.autopandas = True
# Don't print number of rows affected.
%config SqlMagic.feedback = False

# If you set Tools > Settings > Site > Theme to dark, uncomment this line.
# set_dark_mode()
# If you are creating plots to copy-paste into reports, uncomment this line.
# set_light_mode()

## Select a publication from the [Data Registry](https://data.open-contracting.org/) and its field list

In [ ]:
# @title Select the publication to download { display-mode: "form" }

publication_select_box = get_publication_select_box()
publication_select_box

In [ ]:
# @title Extract the list of available fields { display-mode: "form" }

selected_publication = next(entry for entry in get_publications() if entry["label"] == publication_select_box.value)
fields_table = format_coverage(selected_publication.get("coverage", {}))

## Relevance analysis

Use this section to assess if the publication contains the required fields to answer "who bought what from whom, for how much, when and how" for some subset of contracting processes.

Generate a list of the fields published:

In [ ]:
fields_list = fields_table.iloc[:, 0].tolist()

In [ ]:
relevant, result = is_relevant(fields_list)

### Does the publication pass the relevant criterion?

In [ ]:
relevant

### Why?

In [ ]:
result

### Manually check for other fields

If the main OCDS fields are not available to answer the relevant question, you should manually check if others might be used instead. This involves not only checking for the existence of a field but its content too. For example:
- If you cannot answer "who", you could check if they disclose "buyer" or "procuringEntity" roles as part of the parties array.
- If you cannot answer "from whom", you could check if they disclose the "supplier" role as part of the parties array.

If a quick check yields no alternative field, do not spend more time. If you cannot easily find the relevant field, neither will another user.

In [ ]:
fields_table

#### Save the table to a spreadsheet

In [ ]:
spreadsheet_name = input("Enter the name of your spreadsheet:")
save_dataframe_to_sheet(spreadsheet_name, result, "relevant table")